# Phase 1 gate — LaBraM vs hand-crafted EEG (live)

**Prereq:** batch-extract both feature sets first (in a terminal), then run this top to bottom:
```bash
python src/extract_eeg_embeddings.py  --batch --input $SCRATCH/mh/data/mipdb/ --output-dir $SCRATCH/mh/data/processed/
python src/extract_handcrafted_eeg.py --batch --input $SCRATCH/mh/data/mipdb/ --output-dir $SCRATCH/mh/data/processed_handcrafted/
```
Kernel: **Python (mh311)**. The gate question: do frozen LaBraM features beat the old hand-crafted EEG features?

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os
sys.path.insert(0, os.path.expanduser('~/mental-health/src'))
import numpy as np, pandas as pd
from compare_fm_vs_handcrafted import load_pooled, build_xy, evaluate, paired_delta

SCRATCH = os.environ.get('SCRATCH') or f"/scratch/{os.environ['USER']}"
FM_DIR  = f'{SCRATCH}/mh/data/processed'
HC_DIR  = f'{SCRATCH}/mh/data/processed_handcrafted'
LABELS  = os.path.expanduser('~/mental-health/data/labels.csv')
print('fm :', FM_DIR); print('hc :', HC_DIR); print('lbl:', LABELS)

In [ ]:
# Pool windows per subject and align to labels (binary: any DX vs none)
labels_df = pd.read_csv(LABELS)
fm = load_pooled(FM_DIR, '_labram', 'embeddings')
hc = load_pooled(HC_DIR, '_handcrafted', 'features')
subjects, y, X = build_xy({'handcrafted': hc, 'labram': fm}, labels_df, task='binary')
X['fused'] = np.hstack([X['handcrafted'], X['labram']])

In [ ]:
# Paired repeated stratified CV (identical splits for every feature set)
results, per_fold_auc = evaluate(X, y, folds=5, repeats=10, seed=0, task='binary')
for name in ['handcrafted', 'labram', 'fused']:
    a_m, a_ci = results[name]['auc']; b_m, b_ci = results[name]['bacc']
    print(f"{name:<12} AUC {a_m:.3f}+/-{a_ci:.3f}   bal-acc {b_m:.3f}+/-{b_ci:.3f}   dim {X[name].shape[1]}")
delta, ci = paired_delta(per_fold_auc, 'labram', 'handcrafted')
print(f"\nPaired AUC delta (LaBraM - hand-crafted): {delta:+.3f} +/- {ci:.3f}")

In [ ]:
# Visualise AUC with 95% CIs
import matplotlib.pyplot as plt
names = ['handcrafted', 'labram', 'fused']
means = [results[n]['auc'][0] for n in names]
cis   = [results[n]['auc'][1] for n in names]
plt.figure(figsize=(5, 3.2))
plt.bar(names, means, yerr=cis, capsize=6, color=['#888', '#3b7', '#37b'])
plt.axhline(0.5, ls='--', c='r', label='chance')
plt.ylim(0.4, 1.0); plt.ylabel('ROC-AUC'); plt.title('Phase 1: EEG-alone screening')
plt.legend(); plt.tight_layout(); plt.show()